<a href="https://colab.research.google.com/github/1aap-s/ML-Pipeline-Leukaemia/blob/V01/Leukemia_ML_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from imblearn.over_sampling import SMOTE
from scipy.stats import zscore

In [ ]:
#Uploading
from google.colab import files
uploaded = files.upload()

In [ ]:
#Checking uploades
import os
print(os.listdir())

In [ ]:
# 01- Loading csv
# training1.csv: raw values + P/A (1/0)
training = pd.read_csv("training1.csv", header=None)

# actual.csv: lifted manually, first row is patient 1
actual = pd.read_csv("actual.csv", header=None)

print("Training shape:", training.shape)
print("Actual shape:", actual.shape)


In [ ]:
# 02 - Seperate value rows and call rows
value_rows = training.iloc[::2, :]  # 0,2,4,... → value rows
call_rows  = training.iloc[1::2, :] # 1,3,5,... → call rows

# Keep only the first 38 patients for training
value_rows = value_rows.iloc[:38, :]
call_rows  = call_rows.iloc[:38, :]

# Ensure numeric (fix leftover strings like 'call') & fill NaN
value_rows = value_rows.apply(pd.to_numeric, errors='coerce').fillna(0)
call_rows  = call_rows.apply(pd.to_numeric, errors='coerce').fillna(0)

# Interleave columns: val1, call1, val2, call2, ...
X_list = []
for i in range(value_rows.shape[0]):  # for each patient
    patient_row = []
    for j in range(value_rows.shape[1]):
        patient_row.append(value_rows.iloc[i, j])
        patient_row.append(call_rows.iloc[i, j])
    X_list.append(patient_row)

X_df = pd.DataFrame(X_list)
print("Combined feature DataFrame shape:", X_df.shape)


In [ ]:
# 03 - Chronological arrangement
# Extract patient IDs from value_rows
patient_ids = value_rows.iloc[:38, 0].astype(int)

# Sort indices by patient ID
sort_idx = np.argsort(patient_ids)

# Reorder X_df (features) by position
X_df = X_df.iloc[sort_idx].reset_index(drop=True)

# Reorder patient_ids using .iloc
patient_ids = patient_ids.iloc[sort_idx].reset_index(drop=True)

# Align labels from actual.csv
actual_train = actual.iloc[:38].reset_index(drop=True)
actual_train["Label"] = actual_train[1].map({"ALL": 1, "AML": 0})
y = actual_train["Label"].values

# Verify alignment
print("Patient IDs after sorting:", patient_ids.tolist())
print("Actual patient IDs:", actual_train[0].tolist())
print("Alignment correct:", all(patient_ids == actual_train[0].values))

In [ ]:
# 04 - actual.csv: column 0 = patient number, column 1 = cancer type
actual_train = actual.iloc[:38, :]  # first 38 patients

# Map cancer type to 1/0
# ALL → 1, AML → 0
label_map = {'ALL': 1, 'AML': 0}
y_labels = actual_train[1].map(label_map).values

print("Patient labels :", y_labels[:38])
print("Total labels shape:", y_labels.shape)  # should be (38,)

In [ ]:
# For 38 patients
# 05 - Interleave value & call columns for each patient
X_list = []
for i in range(value_rows.shape[0]):  # for each patient
    patient_row = []
    for j in range(value_rows.shape[1]):  # for each gene/parameter
        patient_row.append(value_rows.iloc[i, j])
        patient_row.append(call_rows.iloc[i, j])
    X_list.append(patient_row)

# Create DataFrame
X_df = pd.DataFrame(X_list)
X_df = X_df.astype(float)  # convert everything to numeric
print("Combined feature DataFrame shape:", X_df.shape)  # should be (38, 14260)

In [ ]:
# 06 - Removing duplicate columns
X_df = X_df.loc[:, ~X_df.T.duplicated()]
print("After removing duplicate columns:", X_df.shape)

In [ ]:
# 07 - Handle missing values
X_df = X_df.loc[:, ~X_df.T.duplicated()]
print("After removing duplicate columns:", X_df.shape)

In [ ]:
# 08 - Convert all values to numeric (0/1 for call, numeric for values)
X_df = X_df.apply(pd.to_numeric, errors='coerce')

# Fill any remaining NaN (from conversion errors) with 0
X_df.fillna(0, inplace=True)

print("After conversion, X_df shape:", X_df.shape)
print("Any non-numeric left?", X_df.dtypes.unique())

In [ ]:
# 09 - Remove low variance features
print("Before low-variance removal:", X_df.shape)
selector = VarianceThreshold(threshold=1e-5)  # remove near-constant columns
X_df = pd.DataFrame(selector.fit_transform(X_df))
print("After low-variance removal:", X_df.shape)

In [ ]:
# 10 - Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

In [ ]:
# 11 - Outskirting
# Assume X_df is your numeric DataFrame (38 patients x features)
# Compute z-score column-wise
z_scores = zscore(X_df, axis=0)

# Copy for capping
X_df_capped = X_df.copy()

# Compute mean and std for each column
col_mean = X_df.mean()
col_std  = X_df.std()

# Cap values at mean ± 3*std (column-wise)
X_df_capped = X_df_capped.clip(lower=(col_mean - 3*col_std), upper=(col_mean + 3*col_std), axis=1)

print("After outskirting (capping):", X_df_capped.shape)

In [ ]:
# 12 - SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y)

In [ ]:
# 13 - Final output: lists per patient

# X_res = features after SMOTE
# y_res = labels after SMOTE

# Convert X_res to list of lists
X_final_list = X_res.tolist()  # each sublist is one patient: [val1, call1, val2, call2, ...]

# Convert y_res to plain list
y_final_list = y_res.tolist()  # 0 = AML, 1 = ALL

print(f"Number of patients after SMOTE: {len(X_final_list)}")
print(f"Number of labels: {len(y_final_list)}")


In [ ]:
# 14 - Quick check: first patient
print("First patient data:", X_final_list[26])
print("First patient label:", y_final_list[26])

In [ ]:
# 15 - Visualize first two features

plt.figure(figsize=(6,4))
plt.scatter(X_res[:,0], X_res[:,1], c=y_res, cmap='coolwarm', s=50)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Scatter plot of first two features")
plt.colorbar(label="ALL(1)/AML(0)")
plt.show()